In [31]:
import pandas as pd
import numpy as np
import matplotlib as mpl
mpl.rc('font', family = 'Malgun Gothic')
import matplotlib.pyplot as plt


In [32]:
import os
os.listdir('/content')

['.config', '거래대금.xlsx', '수정시가.xlsx', '52주신고가.xlsx', '종가.xlsx', 'sample_data']

In [33]:
종가52주신고가 = pd.read_excel('52주신고가.xlsx', sheet_name='52주신고가', index_col=0)
수정시가 = pd.read_excel('수정시가.xlsx', sheet_name='수정시가', index_col=0)
종가 = pd.read_excel('종가.xlsx', sheet_name='종가', index_col=0)
거래대금 = pd.read_excel('거래대금.xlsx', sheet_name='거래대금', index_col=0)


In [34]:
import os
os.listdir('/content')

['.config', '거래대금.xlsx', '수정시가.xlsx', '52주신고가.xlsx', '종가.xlsx', 'sample_data']

In [35]:
import pandas as pd
import numpy as np

In [36]:
def load_price_like(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name, header=0)

    # 첫 두 컬럼: Symbol, 종목명
    df = df.rename(columns={df.columns[0]: 'Symbol', df.columns[1]: 'Name'})
    df['Symbol'] = df['Symbol'].astype(str).str.strip()

    # 종목코드를 인덱스로
    df = df.set_index('Symbol')

    # 종목명 컬럼 제거
    df = df.drop(columns=['Name'])

    # 날짜 컬럼만 남기기
    new_cols = pd.to_datetime(df.columns, errors='coerce')
    valid = ~pd.isna(new_cols)
    df = df.loc[:, valid]
    df.columns = new_cols[valid]

    return df

거래대금 = load_price_like('거래대금.xlsx', '거래대금')
종가 = load_price_like('종가.xlsx', '종가')
수정시가 = load_price_like('수정시가.xlsx', '수정시가')

/tmp/ipykernel_4514/3619003044.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  new_cols = pd.to_datetime(df.columns, errors='coerce')
/tmp/ipykernel_4514/3619003044.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  new_cols = pd.to_datetime(df.columns, errors='coerce')
/tmp/ipykernel_4514/3619003044.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  new_cols = pd.to_datetime(df.columns, errors='coerce')


In [37]:
raw_52 = pd.read_excel('52주신고가.xlsx', sheet_name='52주신고가', header=None)
raw_52.head(20)

,0,1,2,3,4,5,6,7,8,9,...,4117,4118,4119,4120,4121,4122,4123,4124,4125,4126
0,Refresh,Last Updated: 2026-03-23 14:35:57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,달력기준,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,코드 포트폴리오,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,아이템 포트폴리오,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,출력주기,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,비영업일,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,주말포함,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,기간,20250901,최근일자(20260320),NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,코드,A005930,A000660,A005380,A373220,A402340,A207940,A034020,A012450,A000270,...,A334700,A465620,Q570070,A412560,Q580023,Q700018,Q500070,A306520,Q500038,A301410
9,코드명,삼성전자,SK하이닉스,현대차,LG에너지솔루션,SK스퀘어,삼성바이오로직스,두산에너빌리티,한화에어로스페이스,기아,...,RISE 팔라듐선물인버스(H),ACE 미국빅테크TOP7 Plus인버스(합성),한투 인버스 2X 플래티넘 선물 ETN,TIGER BBIG레버리지,KB 인버스 2X 금 선물 ETN(H),하나 인버스 2X 코스닥150 선물 ETN,신한 인버스 2X 코스피 200 선물 ETN,HANARO 200선물인버스,신한 인버스 2X 금 선물 ETN,PLUS 코스닥150선물인버스


In [38]:
codes = raw_52.iloc[8, 1:]         # 엑셀 9행
dates = pd.to_datetime(raw_52.iloc[14:, 0], errors='coerce')   # 엑셀 15행부터 날짜
data = raw_52.iloc[14:, 1:]

# 날짜가 정상인 행만 남기기
valid_rows = ~dates.isna()
dates = dates[valid_rows]
data = data.loc[valid_rows]

# 컬럼명을 종목코드로 설정
data.columns = codes.astype(str).str.strip()
data.index = dates

# 숫자로 변환
종가52주신고가 = data.apply(pd.to_numeric, errors='coerce')

# 다른 파일들과 맞추기 위해 전치
종가52주신고가 = 종가52주신고가.T
종가52주신고가.index.name = 'Symbol'

In [39]:
# 공통 날짜
common_dates = 거래대금.columns.intersection(종가.columns).intersection(수정시가.columns).intersection(종가52주신고가.columns)

거래대금 = 거래대금[common_dates]
종가 = 종가[common_dates]
수정시가 = 수정시가[common_dates]
종가52주신고가 = 종가52주신고가[common_dates]

# 공통 종목
common_symbols = 거래대금.index.intersection(종가.index).intersection(수정시가.index).intersection(종가52주신고가.index)

거래대금 = 거래대금.loc[common_symbols]
종가 = 종가.loc[common_symbols]
수정시가 = 수정시가.loc[common_symbols]
종가52주신고가 = 종가52주신고가.loc[common_symbols]

In [40]:
print(거래대금.shape)
print(종가.shape)
print(수정시가.shape)
print(종가52주신고가.shape)

print(거래대금.iloc[:3, :3])
print(종가52주신고가.iloc[:3, :3])

(2627, 118)
(2627, 118)
(2627, 118)
(2627, 118)
           2025-09-22    2025-09-23    2025-09-24
Symbol                                           
A005930  2.288084e+12  2.000621e+12  1.550295e+12
A000660  1.541471e+12  1.186487e+12  1.482037e+12
A005380  1.085988e+11  1.036958e+11  8.348961e+10
0        2025-09-22  2025-09-23  2025-09-24
Symbol                                     
A005930     83500.0     84700.0     85400.0
A000660    353000.0    361000.0    361000.0
A005380    259000.0    259000.0    259000.0


In [44]:
results = []

fridays = [d for d in 종가.columns if pd.to_datetime(d).weekday() == 4]

for d in fridays:
    mask = 종가[d] >= 종가52주신고가[d]
    candidates = 거래대금[d][mask].dropna().sort_values(ascending=False).head(10)

    for stock, value in candidates.items():
        results.append([d, stock, value])

screening_result = pd.DataFrame(results, columns=['기준일', '종목', '거래대금'])
screening_result

,기준일,종목,거래대금
0,2025-09-26,A060250,3.955150e+11
1,2025-09-26,A376900,1.483667e+11
2,2025-09-26,A270660,1.004856e+11
3,2025-09-26,A0010V0,9.600218e+10
4,2025-09-26,A476060,6.162168e+10
...,...,...,...
245,2026-03-20,A036930,4.560585e+11
246,2026-03-20,A417200,3.201753e+11
247,2026-03-20,A006360,2.937685e+11
248,2026-03-20,A006910,2.832927e+11


In [45]:
results = []

for d in fridays:
    mask = 종가[d] >= 종가52주신고가[d]
    candidates = 거래대금[d][mask].dropna().sort_values(ascending=False).head(10)

    for stock, value in candidates.items():
        results.append([d, stock, value])

screening_result = pd.DataFrame(results, columns=['기준일', '종목', '거래대금'])
screening_result

,기준일,종목,거래대금
0,2025-09-26,A060250,3.955150e+11
1,2025-09-26,A376900,1.483667e+11
2,2025-09-26,A270660,1.004856e+11
3,2025-09-26,A0010V0,9.600218e+10
4,2025-09-26,A476060,6.162168e+10
...,...,...,...
245,2026-03-20,A036930,4.560585e+11
246,2026-03-20,A417200,3.201753e+11
247,2026-03-20,A006360,2.937685e+11
248,2026-03-20,A006910,2.832927e+11


In [46]:
screening_result.groupby('기준일').size()

,0
기준일,
2025-09-26,10
2025-10-10,10
2025-10-17,10
2025-10-24,10
2025-10-31,10
2025-11-07,10
2025-11-14,10
2025-11-21,10
2025-11-28,10


In [47]:
screening_result.groupby('기준일').head(10)

,기준일,종목,거래대금
0,2025-09-26,A060250,3.955150e+11
1,2025-09-26,A376900,1.483667e+11
2,2025-09-26,A270660,1.004856e+11
3,2025-09-26,A0010V0,9.600218e+10
4,2025-09-26,A476060,6.162168e+10
...,...,...,...
245,2026-03-20,A036930,4.560585e+11
246,2026-03-20,A417200,3.201753e+11
247,2026-03-20,A006360,2.937685e+11
248,2026-03-20,A006910,2.832927e+11


In [48]:
screening_result['거래대금'] = screening_result['거래대금'].map(lambda x: f"{x:,.0f}")
screening_result

,기준일,종목,거래대금
0,2025-09-26,A060250,"395,514,997,285"
1,2025-09-26,A376900,"148,366,749,425"
2,2025-09-26,A270660,"100,485,612,505"
3,2025-09-26,A0010V0,"96,002,181,000"
4,2025-09-26,A476060,"61,621,681,700"
...,...,...,...
245,2026-03-20,A036930,"456,058,520,850"
246,2026-03-20,A417200,"320,175,295,525"
247,2026-03-20,A006360,"293,768,533,775"
248,2026-03-20,A006910,"283,292,728,905"


In [51]:
last_friday = fridays[-1]
print("가장 최근 금요일:", last_friday)

mask = 종가[last_friday] >= 종가52주신고가[last_friday] * 0.90
last_candidates = 거래대금[last_friday][mask].dropna().sort_values(ascending=False).head(20)

last_week_result = pd.DataFrame({
    '종목': last_candidates.index,
    '거래대금': last_candidates.values
})

last_week_result['거래대금'] = last_week_result['거래대금'].map(lambda x: f"{x:,.0f}")
last_week_result

가장 최근 금요일: 2026-03-20 00:00:00


,종목,거래대금
0,A005930,"7,019,725,077,866"
1,A000660,"3,465,653,176,172"
2,A047040,"1,783,847,120,795"
3,A001510,"1,165,837,557,868"
4,A034020,"883,741,173,750"
5,A032820,"730,654,488,075"
6,A475150,"646,619,988,650"
7,A000250,"573,272,097,000"
8,A036930,"456,058,520,850"
9,A322000,"339,008,113,250"


In [65]:
# 1) 이 시트(data)에서 종목코드-종목명 매핑표 만들기
name_map = data.iloc[:, [0, 1]].copy()
name_map.columns = ['Symbol', 'Name']

# 2) 코드/이름 정리
name_map['Symbol'] = name_map['Symbol'].astype(str).str.strip()
name_map['Name'] = (
    name_map['Name']
    .astype(str)
    .str.replace('수정시가(원)', '', regex=False)
    .str.strip()
)

# 3) 최근 금요일
fridays = [d for d in 종가.columns if pd.to_datetime(d).weekday() == 4]
last_friday = fridays[-1]
print("가장 최근 금요일:", last_friday)

# 4) 해당 금요일 기준 후보 추출
mask = 종가[last_friday] >= 종가52주신고가[last_friday] * 0.90
last_candidates = 거래대금[last_friday][mask].dropna().sort_values(ascending=False).head(20)

# 5) 결과 테이블
last_week_result = pd.DataFrame({
    '종목': last_candidates.index,
    '거래대금': last_candidates.values
})

# 6) 종목 코드 형식 통일
last_week_result['종목'] = last_week_result['종목'].astype(str).str.strip()

# 7) 종목명 붙이기
last_week_result = last_week_result.drop(columns=['Name'], errors='ignore')
last_week_result = last_week_result.merge(
    name_map[['Symbol', 'Name']],
    left_on='종목',
    right_on='Symbol',
    how='left'
)

# 8) 컬럼 정리
last_week_result = last_week_result[['종목', 'Name', '거래대금']]

# 9) 거래대금 포맷팅
last_week_result['거래대금'] = last_week_result['거래대금'].map(lambda x: f"{x:,.0f}")

last_week_result

가장 최근 금요일: 2026-03-20 00:00:00


,종목,Name,거래대금
0,A005930,NaN,"7,019,725,077,866"
1,A000660,NaN,"3,465,653,176,172"
2,A047040,NaN,"1,783,847,120,795"
3,A001510,NaN,"1,165,837,557,868"
4,A034020,NaN,"883,741,173,750"
5,A032820,NaN,"730,654,488,075"
6,A475150,NaN,"646,619,988,650"
7,A000250,NaN,"573,272,097,000"
8,A036930,NaN,"456,058,520,850"
9,A322000,NaN,"339,008,113,250"


In [69]:
check_codes = ['A005930', 'A000660']  # 삼성전자, SK하이닉스

check_df = pd.DataFrame({
    '종가': 종가[last_friday].loc[check_codes],
    '52주신고가파일값': 종가52주신고가[last_friday].loc[check_codes],
    '조건통과여부': (종가[last_friday].loc[check_codes] >= 종가52주신고가[last_friday].loc[check_codes])
})

check_df

,종가,52주신고가파일값,조건통과여부
Symbol,,,
A005930,199400,218000.0,False
A000660,1007000,1099000.0,False


In [70]:
# 가장 최근 금요일 다시 확인
last_friday = fridays[-1]
print("가장 최근 금요일:", last_friday)

# 조건: 종가가 52주 신고가 값 이상인 종목만
mask = 종가[last_friday] >= 종가52주신고가[last_friday]

# 조건 통과 종목 수 확인
print("조건 통과 종목 수:", mask.sum())

# 거래대금 상위 10개
last_candidates = 거래대금[last_friday][mask].dropna().sort_values(ascending=False).head(20)

# 다시 결과표 생성
last_week_result = pd.DataFrame({
    '종목': last_candidates.index,
    '거래대금': last_candidates.values
})

# 종목명 붙이기
last_week_result = last_week_result.merge(name_map, left_on='종목', right_on='Symbol', how='left')
last_week_result = last_week_result[['종목', 'Name', '거래대금']]

# 보기 좋게
last_week_result['거래대금'] = last_week_result['거래대금'].map(lambda x: f"{x:,.0f}")

last_week_result

가장 최근 금요일: 2026-03-20 00:00:00
조건 통과 종목 수: 164


,종목,Name,거래대금
0,A047040,NaN,"1,783,847,120,795"
1,A001510,NaN,"1,165,837,557,868"
2,A034020,NaN,"883,741,173,750"
3,A475150,NaN,"646,619,988,650"
4,A000250,NaN,"573,272,097,000"
5,A036930,NaN,"456,058,520,850"
6,A417200,NaN,"320,175,295,525"
7,A006360,NaN,"293,768,533,775"
8,A006910,NaN,"283,292,728,905"
9,A178320,NaN,"239,115,834,675"


In [71]:
explain_df = pd.DataFrame({
    '종가': 종가[last_friday].loc[last_candidates.index],
    '52주신고가값': 종가52주신고가[last_friday].loc[last_candidates.index],
    '거래대금': 거래대금[last_friday].loc[last_candidates.index]
})

explain_df['조건통과여부'] = explain_df['종가'] >= explain_df['52주신고가값']

# 인덱스(Symbol)를 컬럼으로 빼기
explain_df = explain_df.reset_index()
explain_df = explain_df.rename(columns={'Symbol': '종목코드'})

# 종목명 붙이기
explain_df = explain_df.merge(name_map, left_on='종목코드', right_on='Symbol', how='left')

# 필요한 컬럼만 정리
explain_df = explain_df[['종목코드', 'Name', '종가', '52주신고가값', '거래대금', '조건통과여부']]

# 거래대금 기준 정렬
explain_df = explain_df.sort_values('거래대금', ascending=False)

explain_df


,종목코드,Name,종가,52주신고가값,거래대금,조건통과여부
0,A047040,NaN,19110,19110.0,1783847120795,True
1,A001510,NaN,2585,2585.0,1165837557868,True
2,A034020,NaN,109600,109600.0,883741173750,True
3,A475150,NaN,59900,59900.0,646619988650,True
4,A000250,NaN,907000,907000.0,573272097000,True
5,A036930,NaN,72800,72800.0,456058520850,True
6,A417200,NaN,25300,25300.0,320175295525,True
7,A006360,NaN,31800,31800.0,293768533775,True
8,A006910,NaN,12850,12850.0,283292728905,True
9,A178320,NaN,48000,48000.0,239115834675,True


# 섹터별로 하나씩 뽑기

In [76]:
import pandas as pd

In [84]:
sector_df = pd.read_csv('RFM-11-sector.csv')
sector_df.head()

,섹터코드,섹터명,종목코드,종목명
0,En,에너지,A0001A0,덕양에너젠
1,En,에너지,A000440,중앙에너비스
2,En,에너지,A002960,한국쉘석유
3,En,에너지,A003650,미창석유
4,En,에너지,A006120,SK디스커버리


In [85]:
sector_df = sector_df[['종목코드', '종목명', '섹터명']].copy()
sector_df['종목코드'] = sector_df['종목코드'].astype(str).str.strip()
sector_df.head()

,종목코드,종목명,섹터명
0,A0001A0,덕양에너젠,에너지
1,A000440,중앙에너비스,에너지
2,A002960,한국쉘석유,에너지
3,A003650,미창석유,에너지
4,A006120,SK디스커버리,에너지


In [86]:
last_week_result['종목'] = last_week_result['종목'].astype(str).str.strip()

last_week_sector = last_week_result.merge(
    sector_df,
    left_on='종목',
    right_on='종목코드',
    how='left'
)

last_week_sector[['종목', 'Name', '섹터명', '거래대금']]

,종목,Name,섹터명,거래대금
0,A047040,NaN,산업,"1,783,847,120,795"
1,A001510,NaN,금융,"1,165,837,557,868"
2,A034020,NaN,산업,"883,741,173,750"
3,A475150,NaN,유틸리티,"646,619,988,650"
4,A000250,NaN,헬스케어,"573,272,097,000"
5,A036930,NaN,정보기술,"456,058,520,850"
6,A417200,NaN,산업,"320,175,295,525"
7,A006360,NaN,산업,"293,768,533,775"
8,A006910,NaN,산업,"283,292,728,905"
9,A178320,NaN,정보기술,"239,115,834,675"


In [87]:
# 종목명만 저 섹터 시트에서 다시 붙이기
sector_df['종목코드'] = sector_df['종목코드'].astype(str).str.strip()
sector_df['종목명'] = sector_df['종목명'].astype(str).str.strip()
last_week_sector['종목'] = last_week_sector['종목'].astype(str).str.strip()

last_week_sector = last_week_sector.drop(columns=['종목명'], errors='ignore')

last_week_sector = last_week_sector.merge(
    sector_df[['종목코드', '종목명']],
    left_on='종목',
    right_on='종목코드',
    how='left'
)

last_week_sector = last_week_sector.drop(columns=['종목코드'], errors='ignore')

last_week_sector[['종목', '종목명', '섹터명', '거래대금']]

,종목,종목명,섹터명,거래대금
0,A047040,대우건설,산업,"1,783,847,120,795"
1,A001510,SK증권,금융,"1,165,837,557,868"
2,A034020,두산에너빌리티,산업,"883,741,173,750"
3,A475150,SK이터닉스,유틸리티,"646,619,988,650"
4,A000250,삼천당제약,헬스케어,"573,272,097,000"
5,A036930,주성엔지니어링,정보기술,"456,058,520,850"
6,A417200,LS머트리얼즈,산업,"320,175,295,525"
7,A006360,GS건설,산업,"293,768,533,775"
8,A006910,보성파워텍,산업,"283,292,728,905"
9,A178320,서진시스템,정보기술,"239,115,834,675"


# 섹터별 비중 2배 넘으면 안됨